# Notebook 03 — Temporal Deep Models on DAIC-WOZ / E-DAIC

### Why this notebook exists
Notebook 02 showed that **session averages** of facial features carry no detectable depression
signal (permutation p ≈ 0.6 on both corpora). Averages erase *how the face moves over time*,
and that is where depression is expected to show: slowed, flattened facial dynamics. This
notebook tests the hypothesis directly: **does modelling the second-by-second sequence beat the
averages, under the same leakage-free protocol?**

### Input streams (1 step = 1 second)
| Stream | Source | Datasets |
|---|---|---|
| `au` | Action Units + gaze + head pose (+ tracking-quality channel), averaged per second from 30 fps | DAIC-WOZ (CLNF), E-DAIC (OpenFace 2.1) |
| `cnn` | **Deep CNN features of the face images** (`densenet201`, 1 row/s), PCA → 64 dims fitted on training people only | E-DAIC |
| `au+cnn` | both, concatenated per second | E-DAIC |

### Models — all on identical subject-independent folds
| Model | Uses temporal order? | Role |
|---|---|---|
| `functionals` + logistic | no | the Notebook-2b-style baseline |
| `framepool` + attention | no | **ablation**: deep network, but each second processed alone |
| `gru` + attention | yes | recurrent temporal model |
| `tcn` + attention | yes (61 s receptive field) | **proposed** dilated temporal CNN |

If `tcn` beats `framepool`, the gain comes from temporal context and not just from "using deep

> **Needs PyTorch.** In a cell: `%pip install torch` (CPU is fine; a GPU makes it ~10× faster).
> First run builds a sequence cache from your local drive (slow, once). See §1 for runtime settings.

## 1. Config

In [ ]:
import os, re, glob, json, math, time, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold, train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, mean_squared_error, mean_absolute_error
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

DAIC_ROOT  = r"/path/to/DAIC-WOZ"
EDAIC_ROOT = r"/path/to/E-DAIC"
CACHE_DIR  = "outputs/seq_cache"      # ~1 GB for E-DAIC densenet features
OUT        = "outputs/temporal"
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT, exist_ok=True)

CNN_FEATURE  = "densenet201"   # "densenet201", "vgg16" (2x larger), "resnet" (.mat, 30 fps), or None
CNN_MAT_FPS  = 30              # frames per second of the ResNet .mat features (one row per video frame)
CNN_SCALE    = "standardize"   # "standardize" (per-dimension z-score before PCA; used for all main runs)
                               # or "center" (pre-declared variant: subtract mean only) - run for ALL embeddings
CNN_PCA_DIM  = 64
CONF_THRESHOLD = 0.0

# --- model / training ---
CROP_LEN = 180            # seconds per training crop (3 min)
CROPS_PER_SUBJECT = 8     # random crops per person per epoch
BATCH, EPOCHS, PATIENCE = 32, 30, 6
LR, WD, HIDDEN, DROPOUT = 1e-3, 1e-3, 64, 0.3
LAMBDA_SEV = 0.5          # weight of the PHQ-severity head (multi-task)

# --- evaluation ---
N_FOLDS, N_REPEATS, N_SEEDS = 5, 1, 2
N_BOOT, SEED = 2000, 42
CONFIGS = [("functionals", "au"), ("framepool", "au"), ("gru", "au"), ("tcn", "au"),
           ("functionals", "cnn"), ("framepool", "cnn"), ("tcn", "cnn"),
           ("tcn", "au+cnn")]
BASELINE = ("functionals", "au")      # every model is compared against this
IMP_CONFIG = ("tcn", "au")            # model used for cue importance + attention plots (CV)
ATT_CONFIG = ("tcn", "cnn")           # model whose attention is plotted on the official test set (§10)
SKIP_CV = False   # True = skip the cross-validation (§8) and run only the official split (§10); fast

ONLY_DATASETS = None   # e.g. ["E-DAIC (cnn subset)"] to run just that one and skip finished runs
QUICK = False   # True = ~10-minute pipeline check (fewer folds/epochs). Never report QUICK numbers.
if QUICK:
    EPOCHS, N_FOLDS, N_SEEDS, CROPS_PER_SUBJECT, N_BOOT = 4, 3, 1, 4, 300

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
INK, INK2, GRID, BLUE, ORANGE = "#0b0b0b", "#52514e", "#e6e5e0", "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "figure.dpi": 110})
print("device:", DEVICE, "| QUICK:", QUICK, "| CNN stream:", CNN_FEATURE)

## 2. File discovery (same rules as Notebook 0, including the E-DAIC nested-folder fix)

In [ ]:
META = {"frame", "face_id", "timestamp", "confidence", "success", "timer"}
AU_R = re.compile(r"^AU\d+_r$"); AU_C = re.compile(r"^AU\d+_c$")
SKIP = ("hog", "covarep", "formant", "transcript", "audio", "boaw", "bovw", "egemaps",
        "mfcc", "densenet", "resnet", "vgg", "deepspectrum", "bert", "mel")

def participant_dirs(root):
    out = []
    if not os.path.isdir(root): return out
    for dp, dn, fn in os.walk(root):
        if re.fullmatch(r"\d{3,4}_P", os.path.basename(dp)):
            out.append(dp); dn[:] = []           # never descend into a nested NNN_P copy
    return sorted(out)

def pid_of(pdir): return int(re.match(r"(\d+)_P", os.path.basename(pdir)).group(1))

def _headers(path):
    try: return [c.strip() for c in pd.read_csv(path, nrows=0, skipinitialspace=True).columns]
    except Exception: return []

def feature_files(pdir):
    out = []
    for f in glob.glob(os.path.join(pdir, "**", "*.*"), recursive=True):
        n = os.path.basename(f).lower()
        if n.rsplit(".", 1)[-1] in ("txt", "csv") and not any(s in n for s in SKIP): out.append(f)
    return out

def classify(path, cols):
    name = os.path.basename(path).lower()
    if any(AU_R.match(c) or AU_C.match(c) for c in cols): return "au"
    if "feature" in name: return "landmark"
    if "gaze" in name or any(c.startswith("gaze") for c in cols): return "gaze"
    if ("pose" in name or any(c in ("Tx","Ty","Tz","Rx","Ry","Rz") for c in cols)
            or any(c.startswith("pose_") for c in cols)): return "pose"
    return "other"

def fam_of(kind, c):
    if AU_R.match(c) or AU_C.match(c): return "AU"
    if kind == "gaze" or c.startswith("gaze"): return "gaze"
    return "pose"

def to_seconds(ts):
    t = pd.to_numeric(ts, errors="coerce")
    if t.isna().mean() > 0.5:                     # e.g. "00:01:02.500"
        t = pd.to_timedelta(ts.astype(str), errors="coerce").dt.total_seconds()
    return np.floor(t.values.astype(float))
print("discovery functions ready")

## 3. Labels (official splits)

In [ ]:
def _read_split(path, split, ren):
    d = pd.read_csv(path); d.columns = [c.strip() for c in d.columns]; d = d.rename(columns=ren)
    for c in ["phq_binary", "phq_score"]:
        if c not in d: d[c] = np.nan
    d["split"] = split
    return d[["pid", "phq_binary", "phq_score", "gender", "split"]]

def load_daic_labels(root):
    ren = {"Participant_ID": "pid", "participant_ID": "pid", "PHQ8_Binary": "phq_binary",
           "PHQ8_Score": "phq_score", "Gender": "gender"}
    fr = [_read_split(g[0], s, ren) for s in ["train", "dev", "test"]
          if (g := glob.glob(os.path.join(root, f"{s}_split*.csv")))]
    if not fr: return pd.DataFrame()
    lab = pd.concat(fr, ignore_index=True)
    lab["gender"] = lab["gender"].map({1: "F", 0: "M"}).fillna(lab["gender"].astype(str))
    return lab

def load_edaic_labels(root):
    base = os.path.join(root, "labels2019", "labels")
    if not os.path.isdir(base): base = os.path.join(root, "labels")
    ren = {"Participant_ID": "pid", "PHQ_Binary": "phq_binary", "PHQ_Score": "phq_score", "Gender": "gender"}
    fr = [_read_split(os.path.join(base, f"{s}_split.csv"), s, ren) for s in ["train", "dev", "test"]
          if os.path.exists(os.path.join(base, f"{s}_split.csv"))]
    return pd.concat(fr, ignore_index=True) if fr else pd.DataFrame()

LABELS = {}
for tag, root, fn in [("DAIC", DAIC_ROOT, load_daic_labels), ("E-DAIC", EDAIC_ROOT, load_edaic_labels)]:
    if os.path.isdir(root):
        lab = fn(root)
        if len(lab):
            lab = lab[lab.phq_binary.notna()].drop_duplicates("pid").reset_index(drop=True)
            lab["phq_binary"] = lab.phq_binary.astype(int)
            LABELS[tag] = lab
            print(f"{tag}: {len(lab)} labelled | splits {dict(lab.split.value_counts())} | depressed {lab.phq_binary.sum()}")

## 4. Build per-second sequences (cached)

For each participant: read only the needed columns, blank out untracked frames, average every
second, forward-fill gaps, and add a `track:ok` channel (fraction of tracked frames in that
second). The CNN stream is read at its native 1 row/second. Results are cached in `CACHE_DIR`,
so this slow step runs once; delete the cache folder to rebuild.

In [ ]:
def read_cols(path, keep):
    return pd.read_csv(path, skipinitialspace=True, usecols=lambda c: c.strip() in keep).rename(columns=str.strip)

def au_stream(pdir):
    parts = []
    for f in feature_files(pdir):
        cols = _headers(f)
        if not cols or "timestamp" not in cols: continue
        kind = classify(f, cols)
        if kind not in ("au", "gaze", "pose"): continue
        if kind == "au":
            ch = [c for c in cols if AU_R.match(c) or AU_C.match(c) or c.startswith("gaze_")
                  or c.startswith("pose_R") or c.startswith("pose_T")]
        else:
            ch = [c for c in cols if c not in META and not c.startswith("Unnamed")]
        if not ch: continue
        meta = [c for c in ("timestamp", "confidence", "success") if c in cols]
        df = read_cols(f, set(ch + meta))
        ok = np.ones(len(df), bool)
        if "success" in df: ok &= pd.to_numeric(df["success"], errors="coerce").values == 1
        if CONF_THRESHOLD > 0 and "confidence" in df:
            ok &= pd.to_numeric(df["confidence"], errors="coerce").values >= CONF_THRESHOLD
        X = df[ch].apply(pd.to_numeric, errors="coerce")
        X.loc[~ok, :] = np.nan
        X.columns = [f"{fam_of(kind, c)}:{c}" for c in ch]
        if kind == "au": X["track:ok"] = ok.astype(float)
        X["__sec"] = to_seconds(df["timestamp"])
        parts.append(X.groupby("__sec").mean())
    if not parts: return None
    S = pd.concat(parts, axis=1); S = S.loc[:, ~S.columns.duplicated()]
    S = S[S.index >= 0]
    return S.reindex(np.arange(0, int(S.index.max()) + 1)).ffill().bfill().fillna(0.0)

def _load_mat(path):
    """ResNet features: one array of shape (frames, dims). Handles MATLAB v5 and v7.3 (HDF5)."""
    arr, err = None, None
    try:
        from scipy.io import loadmat
        m = loadmat(path)
        keys = [k for k in m if not k.startswith("__")]
        arr = m["feature"] if "feature" in m else m[keys[0]]
    except Exception as e1:                          # v7.3 (HDF5) or damaged header
        try:
            import h5py
            with h5py.File(path, "r") as h:
                k = "feature" if "feature" in h else [k for k in h.keys() if not k.startswith("#")][0]
                arr = np.array(h[k]).T
        except Exception as e2:
            err = f"{type(e1).__name__}: {e1} | h5py: {type(e2).__name__}"
    if arr is None:                                  # unreadable file -> skip this participant
        BAD_MAT.append((path, os.path.getsize(path), err))
        return None
    return np.nan_to_num(np.asarray(arr, dtype=np.float32))
BAD_MAT = []

def cnn_stream(pdir):
    if not CNN_FEATURE: return None
    if CNN_FEATURE.lower() == "resnet":              # *_CNN_ResNet.mat, 30 rows per second
        g = glob.glob(os.path.join(pdir, "**", "*_CNN_ResNet.mat"), recursive=True)
        if not g: return None
        X = _load_mat(g[0])
        if X is None or X.ndim != 2 or len(X) < CNN_MAT_FPS: return None
        secs = (np.arange(len(X)) // CNN_MAT_FPS).astype(int)
        starts = np.r_[0, np.flatnonzero(np.diff(secs)) + 1]
        means = np.add.reduceat(X, starts, axis=0) / np.bincount(secs)[:, None]
        return pd.DataFrame(means, columns=[f"neuron_{k}" for k in range(X.shape[1])])
    g = glob.glob(os.path.join(pdir, "**", f"*_{CNN_FEATURE}.csv"), recursive=True)
    if not g: return None
    df = pd.read_csv(g[0])
    neur = [c for c in df.columns if c.startswith("neuron_")]
    tcol = next((c for c in df.columns if c.lower() == "timestamp"), None)
    X = df[neur].astype(np.float32)
    sec = None
    if tcol:                                    # use timeStamp only if it is ~1 row per second
        t = pd.to_numeric(df[tcol], errors="coerce")
        if t.isna().mean() > 0.5:
            t = pd.to_timedelta(df[tcol].astype(str), errors="coerce").dt.total_seconds()
        d = np.nanmedian(np.diff(t.values.astype(float))) if t.notna().sum() > 2 else np.nan
        if np.isfinite(d) and 0.5 <= d <= 2.0:      sec = np.floor(t.values.astype(float))
        elif np.isfinite(d) and 500 <= d <= 2000:   sec = np.floor(t.values.astype(float) / 1000.0)  # ms
    if sec is None: sec = np.arange(len(df), dtype=float)   # fallback: 1 row = 1 second
    sec = sec - np.nanmin(sec)
    X["__sec"] = sec
    S = X.groupby("__sec").mean(); S = S[S.index >= 0]
    return S.reindex(np.arange(0, int(S.index.max()) + 1)).ffill().bfill().fillna(0.0)

def build_stream(tag, root, stream, fn):
    path = os.path.join(CACHE_DIR, f"{tag}_{stream}.npz"); meta = path.replace(".npz", ".json")
    if os.path.exists(path) and os.path.exists(meta):
        z = np.load(path); ch = json.load(open(meta))
        return {int(k[1:]): z[k] for k in z.files}, ch
    want = set(LABELS[tag].pid); seqs, ch = {}, None
    for pdir in tqdm(participant_dirs(root), desc=f"{tag}:{stream}"):
        pid = pid_of(pdir)
        if pid not in want or pid in seqs: continue
        S = fn(pdir)
        if S is None or len(S) < 10: continue
        if ch is None: ch = list(S.columns)
        seqs[pid] = S.reindex(columns=ch, fill_value=0.0).values.astype(np.float16)
    if not seqs: return {}, []
    np.savez(path, **{f"p{k}": v for k, v in seqs.items()}); json.dump(ch, open(meta, "w"))
    return seqs, ch

SEQ = {}
for tag, root in [("DAIC", DAIC_ROOT), ("E-DAIC", EDAIC_ROOT)]:
    if tag not in LABELS: continue
    t0 = time.time()
    au, ch_au = build_stream(tag, root, "au", au_stream)
    cnn, ch_cnn = (build_stream(tag, root, CNN_FEATURE, cnn_stream) if CNN_FEATURE else ({}, []))
    lab = LABELS[tag][LABELS[tag].pid.isin(au)].reset_index(drop=True)
    assert lab.pid.is_unique, "LEAKAGE GUARD: duplicate participant"
    with_cnn = [p for p in lab.pid if p in cnn]
    def pack(l, use_cnn):
        return dict(lab=l, y=l.phq_binary.values, sev=l.phq_score.values.astype(float),
                    au=[au[p] for p in l.pid], ch_au=ch_au,
                    cnn=[cnn[p] for p in l.pid] if use_cnn else None)
    if with_cnn and len(with_cnn) == len(lab):
        SEQ[tag] = pack(lab, True); note = f"cnn {len(ch_cnn)} dims (all subjects)"
    else:
        SEQ[tag] = pack(lab, False)
        note = f"cnn files for {len(with_cnn)} of {len(lab)} subjects"
        if len(with_cnn) >= 30:
            sub = lab[lab.pid.isin(with_cnn)].reset_index(drop=True)
            SEQ[f"{tag} (cnn subset)"] = pack(sub, True)
            note += f" -> extra dataset '{tag} (cnn subset)' with ALL models on those {len(sub)}"
    L = [len(a) for a in SEQ[tag]["au"]]
    print(f"{tag}: {len(lab)} subjects | au {len(ch_au)} channels | {note} | "
          f"length {int(np.median(L))} s median ({min(L)}-{max(L)}) | {time.time()-t0:.0f}s")
    if cnn and len(with_cnn) < len(lab):
        miss = sorted(set(lab.pid) - set(with_cnn))
        print(f"   participants without {CNN_FEATURE}: {miss[:20]}{' ...' if len(miss) > 20 else ''}")
if BAD_MAT:
    print(f"\n{len(BAD_MAT)} unreadable .mat file(s) skipped (damaged/incomplete copy?):")
    for pth, size, err in BAD_MAT[:15]:
        print(f"   {pth}  ({size/1e6:.1f} MB)  {err[:90]}")

## 5. Models

In [ ]:
class AttnPool(nn.Module):
    def __init__(s, d):
        super().__init__(); s.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(s, h):
        w = torch.softmax(s.score(h).squeeze(-1), dim=1)
        return (w.unsqueeze(-1) * h).sum(1), w

class Net(nn.Module):
    """arch: 'framepool' (no temporal context) | 'gru' | 'tcn'. Two heads: depressed logit + PHQ (z-scored)."""
    def __init__(s, arch, C, d=HIDDEN, p=DROPOUT):
        super().__init__(); s.arch = arch
        s.inp = nn.Sequential(nn.Dropout(0.1), nn.Linear(C, d), nn.GELU())
        if arch == "tcn":
            s.blocks = nn.ModuleList([nn.Sequential(nn.Conv1d(d, d, 5, padding=2 * dl, dilation=dl),
                                                    nn.BatchNorm1d(d), nn.GELU(), nn.Dropout(p))
                                      for dl in (1, 2, 4, 8)])
        elif arch == "gru":
            s.rnn = nn.GRU(d, d // 2, batch_first=True, bidirectional=True)
        elif arch == "framepool":
            s.mlp = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(p))
        else:
            raise ValueError(arch)
        s.pool = AttnPool(d); s.cls = nn.Linear(d, 1); s.reg = nn.Linear(d, 1)
    def forward(s, x):                       # x: (B, T, C)
        h = s.inp(x)
        if s.arch == "tcn":
            z = h.transpose(1, 2)
            for b in s.blocks: z = z + b(z)  # residual dilated convs
            h = z.transpose(1, 2)
        elif s.arch == "gru":
            h, _ = s.rnn(h)
        else:
            h = s.mlp(h)
        z, w = s.pool(h)
        return s.cls(z).squeeze(-1), s.reg(z).squeeze(-1), w

for a in ["framepool", "gru", "tcn"]:
    n = sum(p.numel() for p in Net(a, 50).parameters())
    print(f"{a:10s} {n/1e3:.0f}k parameters (for 50 input channels)")

## 6. Training utilities

Training uses random **3-minute crops** (many per person per epoch: augmentation for small N).
At test time each person is covered by overlapping crops and their logits are averaged. Early
stopping uses a validation set of training people (never the test fold). Normalisation and
CNN-PCA are fitted on the training people of each fold only.

In [ ]:
def crop(a, L, rng):
    T = len(a)
    if T >= L:
        s = rng.integers(0, T - L + 1); return a[s:s + L]
    return np.resize(a, (L, a.shape[1]))

def windows(a, L):
    T = len(a)
    if T <= L: return [np.resize(a, (L, a.shape[1]))]
    st = list(range(0, T - L + 1, L // 2))
    if st[-1] != T - L: st.append(T - L)
    return [a[s:s + L] for s in st]

@torch.no_grad()
def predict(m, seqs):
    m.eval(); P, R = [], []
    for a in seqs:
        W = torch.tensor(np.stack(windows(a, CROP_LEN)), dtype=torch.float32, device=DEVICE)
        lo, re_, _ = m(W); P.append(lo.mean().item()); R.append(re_.mean().item())
    return np.array(P), np.array(R)

def train_model(arch, Xtr, ytr, sztr, Xva, yva, seed):
    seed_all(seed); rng = np.random.default_rng(seed)
    m = Net(arch, Xtr[0].shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    pw = torch.tensor(max(1.0, (ytr == 0).sum() / max(1, (ytr == 1).sum())), dtype=torch.float32, device=DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, bad, state = np.inf, 0, None
    yva_t = torch.tensor(yva, dtype=torch.float32)
    for ep in range(EPOCHS):
        m.train()
        idx = np.repeat(np.arange(len(Xtr)), CROPS_PER_SUBJECT); rng.shuffle(idx)
        for b in range(0, len(idx), BATCH):
            bi = idx[b:b + BATCH]
            x = np.stack([crop(Xtr[i], CROP_LEN, rng) for i in bi]).astype(np.float32)
            x += rng.normal(0, 0.05, x.shape).astype(np.float32)
            x = torch.tensor(x, device=DEVICE)
            yb = torch.tensor(ytr[bi], dtype=torch.float32, device=DEVICE)
            sb = torch.tensor(sztr[bi], dtype=torch.float32, device=DEVICE)
            lo, re_, _ = m(x)
            mask = torch.isfinite(sb)
            loss = bce(lo, yb) + (LAMBDA_SEV * F.mse_loss(re_[mask], sb[mask]) if mask.any() else 0.0)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        pv, _ = predict(m, Xva)
        vloss = F.binary_cross_entropy_with_logits(torch.tensor(pv, dtype=torch.float32), yva_t,
                                                   pos_weight=pw.cpu()).item()
        if vloss < best - 1e-4:
            best, bad, state = vloss, 0, {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    m.load_state_dict(state); return m

def fold_streams(S, tr_idx):
    """Normalise (and PCA the CNN stream) using training people only. Returns dict stream -> list."""
    out = {}
    A = S["au"]
    samp = np.concatenate([A[i] for i in tr_idx]).astype(np.float32)
    mu, sd = samp.mean(0), samp.std(0) + 1e-6
    out["au"] = [((a.astype(np.float32) - mu) / sd) for a in A]
    if S["cnn"] is not None:
        rng = np.random.default_rng(SEED)
        frames = [S["cnn"][i][rng.choice(len(S["cnn"][i]), min(200, len(S["cnn"][i])), replace=False)]
                  for i in tr_idx]
        Fs = np.concatenate(frames).astype(np.float32)
        cmu = Fs.mean(0)
        csd = (Fs.std(0) + 1e-6) if CNN_SCALE == "standardize" else np.ones(Fs.shape[1], np.float32)
        pca = PCA(n_components=min(CNN_PCA_DIM, Fs.shape[1], len(Fs) - 1), svd_solver="randomized",
                  random_state=SEED).fit((Fs - cmu) / csd)
        Z = [pca.transform((c.astype(np.float32) - cmu) / csd) for c in S["cnn"]]
        zs = np.concatenate([Z[i] for i in tr_idx]); zmu, zsd = zs.mean(0), zs.std(0) + 1e-6
        out["cnn"] = [((z - zmu) / zsd).astype(np.float32) for z in Z]
        out["au+cnn"] = [np.concatenate([a[:min(len(a), len(c))], c[:min(len(a), len(c))]], 1)
                         for a, c in zip(out["au"], out["cnn"])]
    return out

def functionals(seqs):
    f = []
    for a in seqs:
        d = np.abs(np.diff(a, axis=0)).mean(0) if len(a) > 1 else np.zeros(a.shape[1])
        f.append(np.concatenate([a.mean(0), a.std(0), np.percentile(a, 10, 0), np.percentile(a, 90, 0), d]))
    return np.nan_to_num(np.array(f))

def functionals_fit_predict(X, y, sev, tr, te):
    Ftr, Fte = functionals([X[i] for i in tr]), functionals([X[i] for i in te])
    clf = GridSearchCV(Pipeline([("sc", StandardScaler()),
                                 ("lr", LogisticRegression(max_iter=5000, class_weight="balanced"))]),
                       {"lr__C": [0.001, 0.01, 0.1, 1.0]}, scoring="roc_auc",
                       cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).fit(Ftr, y[tr])
    ok = np.isfinite(sev[tr])
    reg = GridSearchCV(Pipeline([("sc", StandardScaler()), ("rg", Ridge())]),
                       {"rg__alpha": [10, 100, 1000, 10000]}, scoring="neg_root_mean_squared_error",
                       cv=KFold(5, shuffle=True, random_state=SEED)).fit(Ftr[ok], sev[tr][ok])
    return clf.predict_proba(Fte)[:, 1], reg.predict(Fte)
print("training utilities ready")

## 7. Interpretation helpers

**Grouped importance:** swap a cue group's channels between test people (keeping each sequence
intact) and measure the AUC drop. **Attention over time:** where in the interview the TCN
looks, averaged for depressed vs non-depressed people.

In [ ]:
def channel_groups(ch):
    g = {}
    for i, c in enumerate(ch):
        fam = c.split(":")[0]; g.setdefault(fam, []).append(i)
        m = re.search(r"(AU\d+)_", c)
        if m: g.setdefault(m.group(1), []).append(i)
    return g

def group_importance(m, X, te, y, ch, rng):
    if len(np.unique(y[te])) < 2: return {}
    base_p, _ = predict(m, [X[i] for i in te]); base = roc_auc_score(y[te], base_p); out = {}
    for g, idx in channel_groups(ch).items():
        perm = rng.permutation(len(te)); Xp = []
        for k, i in enumerate(te):
            a = X[i].copy(); donor = X[te[perm[k]]]
            a[:, idx] = np.resize(donor[:, idx], (len(a), len(idx))); Xp.append(a)
        p, _ = predict(m, Xp); out[g] = base - roc_auc_score(y[te], p)
    return out

@torch.no_grad()
def attention_profiles(m, X, te, y, bins=50):
    m.eval(); prof = []
    for i in te:
        x = torch.tensor(X[i][None], dtype=torch.float32, device=DEVICE)
        _, _, w = m(x); w = w[0].cpu().numpy()
        prof.append((int(y[i]), np.interp(np.linspace(0, len(w) - 1, bins), np.arange(len(w)), w) * len(w)))
    return prof

def plot_attention(prof, title, fname):
    """Mean attention over the interview (time-normalised), depressed vs not; prints a flatness summary."""
    if not prof: return
    x = np.linspace(0, 100, len(prof[0][1]))
    fig, ax = plt.subplots(figsize=(7, 3)); ends = []
    for lab_, col, name in [(1, ORANGE, "depressed"), (0, BLUE, "not depressed")]:
        P = np.array([p for l, p in prof if l == lab_])
        if not len(P): continue
        m_ = P.mean(0); se = P.std(0) / np.sqrt(len(P))
        ax.fill_between(x, m_ - 1.96 * se, m_ + 1.96 * se, color=col, alpha=0.15, lw=0)
        ax.plot(x, m_, color=col, lw=2); ends.append([m_[-1], f"{name} (n={len(P)})"])
        thirds = [P[:, k * len(x) // 3:(k + 1) * len(x) // 3].mean() for k in range(3)]
        peak = np.mean([p.max() for p in P])
        print(f"   {name:13s} n={len(P):3d} | mean attention in 1st/2nd/3rd third: "
              f"{thirds[0]:.2f} / {thirds[1]:.2f} / {thirds[2]:.2f} (1 = uniform) | mean peak {peak:.2f}")
    if len(ends) == 2:                                    # keep the two end labels apart
        gap = 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0])
        lo_, hi_ = sorted(ends, key=lambda e: e[0])
        if hi_[0] - lo_[0] < gap:
            mid = (hi_[0] + lo_[0]) / 2; lo_[0], hi_[0] = mid - gap / 2, mid + gap / 2
    for yv, txt in ends:
        ax.text(x[-1] + 1.5, yv, txt, color=INK, fontsize=8, va="center")
    ax.axhline(1, color=INK2, lw=0.8, ls=":")
    ax.set_xlabel("position in interview (%)"); ax.set_ylabel("relative attention (1 = uniform)")
    ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True); ax.set_xlim(0, 125)
    ax.set_title(title, color=INK, loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, fname), dpi=200); plt.show()
print("interpretation helpers ready")

## 8. Cross-validated comparison (main experiment)

Stratified K-fold over **people** (each person in exactly one test fold per repeat), identical
folds for every model. Runtime on CPU: roughly 30–60 min for DAIC and 1–2 h for E-DAIC with
the default settings (much faster on a GPU). Progress is printed per fold.

In [ ]:
def stream_ok(S, stream): return stream == "au" or S["cnn"] is not None

RUN = {}
for tag, S in SEQ.items():
    if ONLY_DATASETS and tag not in ONLY_DATASETS:
        print(f"skipping {tag} (not in ONLY_DATASETS)"); continue
    if SKIP_CV:
        print(f"SKIP_CV = True -> no cross-validation for {tag}"); continue
    y, sev, n = S["y"], S["sev"], len(S["y"])
    cfgs = [c for c in CONFIGS if stream_ok(S, c[1])]
    oof = {c: np.zeros(n) for c in cfgs}; oofs = {c: np.zeros(n) for c in cfgs}
    base_sev = np.zeros(n); cnt = np.zeros(n); IMP = []; ATT = []
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)
    t_all = time.time()
    for fold, (tr, te) in enumerate(rskf.split(np.zeros(n), y)):
        t0 = time.time()
        tr_in, va = train_test_split(tr, test_size=0.15, stratify=y[tr], random_state=SEED + fold)
        streams = fold_streams(S, tr)
        smu, ssd = np.nanmean(sev[tr_in]), np.nanstd(sev[tr_in]) + 1e-6
        sz = (sev - smu) / ssd
        base_sev[te] += np.nanmean(sev[tr])
        for cfg in cfgs:
            arch, stream = cfg; X = streams[stream]
            if arch == "functionals":
                p, r = functionals_fit_predict(X, y, sev, tr, te)
            else:
                L, R = [], []
                for s in range(N_SEEDS):
                    m = train_model(arch, [X[i] for i in tr_in], y[tr_in], sz[tr_in],
                                    [X[i] for i in va], y[va], SEED + 1000 * fold + s)
                    a, b = predict(m, [X[i] for i in te]); L.append(a); R.append(b)
                    if s == 0 and cfg == IMP_CONFIG:
                        IMP.append(group_importance(m, X, te, y, S["ch_au"], np.random.default_rng(fold)))
                        ATT += attention_profiles(m, X, te, y)
                p = 1 / (1 + np.exp(-np.mean(L, 0))); r = np.mean(R, 0) * ssd + smu
            oof[cfg][te] += p; oofs[cfg][te] += r
        cnt[te] += 1
        print(f"{tag} fold {fold+1}/{N_FOLDS*N_REPEATS} done in {time.time()-t0:.0f}s")
    for c in cfgs: oof[c] /= cnt; oofs[c] /= cnt
    RUN[tag] = dict(cfgs=cfgs, oof=oof, oofs=oofs, base_sev=base_sev / cnt, imp=IMP, att=ATT)
    print(f"{tag}: total {time.time()-t_all:.0f}s")

## 9. Results — every model vs the baseline, with paired bootstrap

`ΔAUC vs baseline` is computed on the **same people** (paired bootstrap), so it is the right
test for "does this model beat the averages?". A model earns a claim only if its ΔAUC CI is
above 0.

In [ ]:
def ccc(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(2 * np.mean((a - a.mean()) * (b - b.mean())) / (a.var() + b.var() + (a.mean() - b.mean()) ** 2 + 1e-12))

def boot(fn, *arrs, n=N_BOOT, binary=None):
    rng = np.random.default_rng(SEED); v = []; N = len(arrs[0])
    for _ in range(n):
        i = rng.integers(0, N, N)
        if binary is not None and len(np.unique(binary[i])) < 2: continue
        v.append(fn(*[a[i] for a in arrs]))
    return np.percentile(v, [2.5, 97.5]), np.array(v)

def auc(y, p): return roc_auc_score(y, p)
def mf1(y, p): return f1_score(y, (p >= 0.5).astype(int), average="macro")

TABLES = {}
for tag, R in RUN.items():
    S = SEQ[tag]; y, sev = S["y"], S["sev"]; ok = np.isfinite(sev)
    pb = R["oof"][BASELINE]; rows = []
    for cfg in R["cfgs"]:
        p, r = R["oof"][cfg], R["oofs"][cfg]
        (alo, ahi), _ = boot(auc, y, p, binary=y)
        (flo, fhi), _ = boot(mf1, y, p, binary=y)
        if cfg != BASELINE:
            (dlo, dhi), dv = boot(lambda a, b, c: auc(a, b) - auc(a, c), y, p, pb, binary=y)
            pval = 2 * min((dv <= 0).mean(), (dv >= 0).mean())
            dtxt = f"{auc(y,p)-auc(y,pb):+.3f} [{dlo:+.3f}, {dhi:+.3f}] p={pval:.3f}"
        else:
            dtxt = "(baseline)"
        rows.append(dict(model=f"{cfg[0]} [{cfg[1]}]",
                         AUC=f"{auc(y,p):.3f} [{alo:.3f}, {ahi:.3f}]",
                         macro_F1=f"{mf1(y,p):.3f} [{flo:.3f}, {fhi:.3f}]",
                         bal_acc=f"{balanced_accuracy_score(y,(p>=.5).astype(int)):.3f}",
                         dAUC_vs_baseline=dtxt,
                         RMSE=f"{mean_squared_error(sev[ok], r[ok])**.5:.2f}",
                         MAE=f"{mean_absolute_error(sev[ok], r[ok]):.2f}",
                         CCC=f"{ccc(sev[ok], r[ok]):.3f}"))
    bs = R["base_sev"]
    rows.append(dict(model="mean-PHQ baseline (severity only)", AUC="0.500 (by definition)", macro_F1="-",
                     bal_acc="0.500", dAUC_vs_baseline="-",
                     RMSE=f"{mean_squared_error(sev[ok], bs[ok])**.5:.2f}",
                     MAE=f"{mean_absolute_error(sev[ok], bs[ok]):.2f}", CCC="0.000 (by definition)"))
    T = pd.DataFrame(rows); TABLES[tag] = T
    T.to_csv(os.path.join(OUT, f"{tag}_temporal_results.csv"), index=False)
    P = S["lab"][["pid", "split", "gender", "phq_binary", "phq_score"]].copy()   # per-person predictions
    for (a_, s_) in R["cfgs"]:
        P[f"p_{a_}_{s_}"] = R["oof"][(a_, s_)]; P[f"sev_{a_}_{s_}"] = R["oofs"][(a_, s_)]
    P.to_csv(os.path.join(OUT, f"{tag}_oof_predictions.csv"), index=False)
    print(f"\n==== {tag}: {N_FOLDS}-fold x {N_REPEATS} repeat(s), {N_SEEDS} seed(s) ====")
    print(T.to_string(index=False))

In [ ]:
for tag, R in RUN.items():
    y = SEQ[tag]["y"]; labels, vals, los, his = [], [], [], []
    for cfg in R["cfgs"]:
        p = R["oof"][cfg]; (lo, hi), _ = boot(auc, y, p, binary=y)
        labels.append(f"{cfg[0]} [{cfg[1]}]"); vals.append(auc(y, p)); los.append(lo); his.append(hi)
    vals, los, his = np.array(vals), np.array(los), np.array(his)
    ypos = np.arange(len(labels))[::-1]            # item i drawn at row ypos[i] (first item on top)
    fig, ax = plt.subplots(figsize=(7, 0.42 * len(labels) + 1.4))
    ax.errorbar(vals, ypos, xerr=[vals - los, his - vals], fmt="o", color=BLUE, ecolor=INK2,
                elinewidth=1, capsize=3, markersize=7)
    ax.axvline(0.5, color=ORANGE, lw=1.2, ls="--")
    ax.set_ylim(-1.2, len(labels) - 0.4)
    ax.text(0.505, -0.95, "chance (0.5)", color=INK2, fontsize=8, va="center")
    ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=9)
    ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel("AUC (95% bootstrap CI), subject-independent CV")
    ax.set_title(f"{tag}: temporal models vs averaged-feature baseline", color=INK, loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"fig_{tag}_model_comparison.png"), dpi=200); plt.show()

## 10. E-DAIC official split (train → dev for early stopping → test)

The number comparable to published AVEC-2019 work. Deep models stop early on the official dev
set; the baseline trains on train + dev.

In [ ]:
OFF_TAG = next((t for t in SEQ if t.startswith("E-DAIC") and SEQ[t]["cnn"] is not None),
               "E-DAIC" if "E-DAIC" in SEQ else None)
if OFF_TAG and (not ONLY_DATASETS or OFF_TAG in ONLY_DATASETS):
    print("official split on:", OFF_TAG)
    S = SEQ[OFF_TAG]; lab = S["lab"]; y, sev = S["y"], S["sev"]
    tr = np.where(lab.split == "train")[0]; va = np.where(lab.split == "dev")[0]; te = np.where(lab.split == "test")[0]
    if len(te) and len(va):
        streams = fold_streams(S, tr); smu, ssd = np.nanmean(sev[tr]), np.nanstd(sev[tr]) + 1e-6
        sz = (sev - smu) / ssd; rows = []; OFF_ATT = []
        OFFPRED = lab.iloc[te][["pid", "gender", "phq_binary", "phq_score"]].reset_index(drop=True).copy()
        for cfg in [c for c in CONFIGS if stream_ok(S, c[1])]:
            arch, stream = cfg; X = streams[stream]
            if arch == "functionals":
                p, r = functionals_fit_predict(X, y, sev, np.concatenate([tr, va]), te)
            else:
                L, Rr = [], []
                for s in range(N_SEEDS):
                    m = train_model(arch, [X[i] for i in tr], y[tr], sz[tr], [X[i] for i in va], y[va], SEED + 7 + s)
                    a, b = predict(m, [X[i] for i in te]); L.append(a); Rr.append(b)
                    if cfg == ATT_CONFIG:                      # attention of every seed, test people only
                        OFF_ATT += attention_profiles(m, X, te, y)
                p = 1 / (1 + np.exp(-np.mean(L, 0))); r = np.mean(Rr, 0) * ssd + smu
            (lo, hi), _ = boot(auc, y[te], p, binary=y[te])
            okt = np.isfinite(sev[te]); OFFPRED[f"p_{arch}_{stream}"] = p; OFFPRED[f"sev_{arch}_{stream}"] = r
            rows.append(dict(model=f"{arch} [{stream}]", AUC=f"{auc(y[te], p):.3f} [{lo:.3f}, {hi:.3f}]",
                             macro_F1=f"{mf1(y[te], p):.3f}",
                             RMSE=f"{mean_squared_error(sev[te][okt], r[okt])**.5:.2f}",
                             CCC=f"{ccc(sev[te][okt], r[okt]):.3f}"))
        base = np.full(okt.sum(), np.nanmean(sev[np.concatenate([tr, va])]))
        rows.append(dict(model="mean-PHQ baseline", AUC="0.500", macro_F1="-",
                         RMSE=f"{mean_squared_error(sev[te][okt], base)**.5:.2f}", CCC="0.000"))
        OFF = pd.DataFrame(rows); OFF.to_csv(os.path.join(OUT, f"{OFF_TAG}_official_split.csv"), index=False)
        OFFPRED.to_csv(os.path.join(OUT, f"{OFF_TAG}_official_test_predictions.csv"), index=False)
        print(f"E-DAIC official split: train {len(tr)} / dev {len(va)} / test {len(te)}")
        print(OFF.to_string(index=False))
        if OFF_ATT:
            print(f"\nattention of {ATT_CONFIG[0]} [{ATT_CONFIG[1]}] on the official test set "
                  f"({N_SEEDS} seed(s) x {len(te)} people):")
            plot_attention(OFF_ATT, f"{OFF_TAG}: where in the interview the {ATT_CONFIG[0]} [{ATT_CONFIG[1]}] attends (official test)",
                           f"fig_{OFF_TAG}_official_attention_{CNN_FEATURE}.png")

## 11. What does the TCN rely on, and when?

In [ ]:
for tag, R in RUN.items():
    if not R["imp"]: continue
    imp = pd.DataFrame(R["imp"]).agg(["mean", "std"]).T.sort_values("mean", ascending=False)
    imp.to_csv(os.path.join(OUT, f"{tag}_tcn_group_importance.csv"))
    fam = imp.loc[[g for g in imp.index if not g.startswith("AU")]]
    aus = imp.loc[[g for g in imp.index if g.startswith("AU")]].head(10)
    print(f"\n{tag} — AUC drop when a cue group is swapped between people (TCN [au]):")
    print(imp.head(15).round(4).to_string())
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1.5]})
    for ax, t, title in [(axes[0], fam, "cue family"), (axes[1], aus, "Action Unit (top 10)")]:
        t = t.iloc[::-1]
        ax.barh(range(len(t)), t["mean"], xerr=t["std"], color=BLUE, height=0.6,
                error_kw={"ecolor": INK2, "lw": 0.8, "capsize": 2})
        ax.set_yticks(range(len(t))); ax.set_yticklabels(t.index, fontsize=8)
        ax.axvline(0, color=INK2, lw=0.8); ax.grid(axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
        ax.set_xlabel("AUC drop"); ax.set_title(title, color=INK, loc="left", fontsize=10)
    fig.suptitle(f"{tag}: cues the temporal model uses", x=0.01, ha="left", color=INK)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"fig_{tag}_tcn_importance.png"), dpi=200); plt.show()

    plot_attention(R["att"], f"{tag}: where in the interview the TCN [au] attends (CV)",
                   f"fig_{tag}_attention_time.png")

## 12. Summary

In [ ]:
summary = {"settings": dict(CROP_LEN=CROP_LEN, EPOCHS=EPOCHS, N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS,
                            N_SEEDS=N_SEEDS, CNN_FEATURE=CNN_FEATURE, CNN_PCA_DIM=CNN_PCA_DIM, CNN_SCALE=CNN_SCALE, QUICK=QUICK),
           "n_subjects": {t: int(len(S["y"])) for t, S in SEQ.items()},
           "auc": {t: {f"{c[0]} [{c[1]}]": round(float(roc_auc_score(SEQ[t]["y"], R["oof"][c])), 4)
                       for c in R["cfgs"]} for t, R in RUN.items()}}
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=2)
print(json.dumps(summary, indent=2)); print("\nSaved to", OUT)